## 6. Filter Bank Common Spatial Patterns (FBCSP)

The baseline CSP+LDA pipeline uses a single 8–30 Hz band. But motor-imagery
signal sits in different frequencies for different people — some in the mu band
(~8–13 Hz), some in beta (~13–30 Hz). A single wide band mixes the informative
sub-band with uninformative ones, diluting the signal.

**FBCSP** (Ang et al., BCI Competition IV winner) addresses this by analyzing
multiple narrow frequency bands and letting the data decide which matter:

1. **Filter bank** — split the signal into narrow bands (here, 7 bands of 4 Hz
   width from 4–32 Hz), covering theta, mu, and beta.
2. **CSP per band** — run CSP *separately* on each band, so each frequency range
   gets its own spatial filters optimized for that band.
3. **Feature selection** — pool the features from all bands, then select the most
   discriminative ones (via mutual information). This automatically identifies
   *which bands carry the signal for this specific subject* — the key advantage
   over single-band CSP.
4. **Classification** — LDA on the selected features.

The core idea: instead of guessing one frequency band, FBCSP tries many and lets
the data pick. This directly targets the subject-to-subject variability found
earlier, where different subjects' signals lived in different frequency ranges.

**Future directions (deep learning):** FBCSP's multi-band spatial-filtering idea
has neural-network descendants worth benchmarking later — **FBCNet** (a deep
learning version of FBCSP's multi-band structure) and **EEGNet** (a compact CNN
noted in reviews for its generalizability). These need more data than the
classical methods, so they are planned as later comparisons once the classical
pipeline and hardware are working.

In [1]:
# Imports for FBCSP work
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.decoding import CSP
from pathlib import Path

# scikit-learn: classifier, pipeline, cross-validation, feature selection
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif

print("mne:", mne.__version__)
print("Imports OK")

mne: 1.12.1
Imports OK


In [ ]:
# Shared configuration
data_dir = Path.home() / "projects" / "bci" / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Standard electrode-position montage, reused throughout
montage = mne.channels.make_standard_montage("standard_1005")

# The filter bank: 7 narrow bands (4 Hz wide) covering theta/mu/beta
filter_bank = [
    (8, 12), (12, 16), (16, 20),
    (20, 24), (24, 28), (28, 32),
]
s
print("Data dir:", data_dir)
print(f"Filter bank: {len(filter_bank)} bands ->", filter_bank)

Data dir: /home/beecki303/projects/bci/data/raw
Filter bank: 6 bands -> [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32)]


In [3]:
def load_subject_raw(subject, runs=(4, 8, 12)):
    """Download + load + standardize a subject's motor imagery runs."""
    fnames = eegbci.load_data(subjects=[subject], runs=list(runs),
                              path=str(data_dir), verbose=False)
    raws = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in fnames]
    raw = mne.concatenate_raws(raws, verbose=False)
    eegbci.standardize(raw)
    raw.set_montage(montage, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    return raw, events, event_id

In [4]:
def make_band_epochs(raw_source, band, events, event_id):
    """Bandpass to one filter-bank band, then epoch the imagery window."""
    lo, hi = band
    r = raw_source.copy()
    r.notch_filter(60.0, picks='eeg', verbose=False)
    r.filter(lo, hi, picks='eeg', verbose=False)
    ep = mne.Epochs(r, events,
                    event_id={'T1': event_id['T1'], 'T2': event_id['T2']},
                    tmin=0.5, tmax=2.5, baseline=None,
                    picks='eeg', preload=True, verbose=False)
    return ep

def fbcsp_features(raw_source, events, event_id, n_csp=4):
    """Run CSP on each band, stack log-variance features across all bands."""
    band_features = []
    for band in filter_bank:
        ep = make_band_epochs(raw_source, band, events, event_id)
        Xb = ep.get_data(copy=False)
        yb = ep.events[:, -1]
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        feats = csp.fit_transform(Xb, yb)
        band_features.append(feats)
    X_fb = np.hstack(band_features)
    return X_fb, yb

In [5]:
# Verify setup: load subject 2 and build the FBCSP feature matrix
raw_s, evs, ev_id = load_subject_raw(2)
X_fb, y_fb = fbcsp_features(raw_s, evs, ev_id, n_csp=4)

print("FBCSP feature matrix shape:", X_fb.shape)
print(f"  = {len(filter_bank)} bands x 4 CSP components = {X_fb.shape[1]} features")
print("Labels shape:", y_fb.shape)

Computing rank from data with rank=None
    Using tolerance 8.5e-05 (2.2e-16 eps * 64 dim * 6e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 7.5e-05 (2.2e-16 eps * 64 dim * 5.3e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 5.1e-05 (2.2e-16 eps * 64 dim * 3.6e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIR

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class FBCSP(BaseEstimator, TransformerMixin):
    """Filter Bank CSP: runs CSP per pre-filtered band, stacks features.
    Expects X as a dict-free 4D array: (trials, bands, channels, times)."""
    def __init__(self, n_csp=4):
        self.n_csp = n_csp
        self.csps = []

    def fit(self, X, y):
        self.csps = []
        for b in range(X.shape[1]):          # loop over bands
            csp = CSP(n_components=self.n_csp, reg=None, log=True, norm_trace=False)
            csp.fit(X[:, b, :, :], y)         # fit CSP on this band's data
            self.csps.append(csp)
        return self

    def transform(self, X):
        feats = [self.csps[b].transform(X[:, b, :, :]) for b in range(X.shape[1])]
        return np.hstack(feats)

def build_band_data(raw_source, events, event_id):
    """Pre-filter into all bands ONCE, return 4D array (trials, bands, channels, times)."""
    band_arrays = []
    for band in filter_bank:
        ep = make_band_epochs(raw_source, band, events, event_id)
        band_arrays.append(ep.get_data(copy=False))
        y = ep.events[:, -1]
    # stack bands into axis 1: (trials, bands, channels, times)
    X4d = np.stack(band_arrays, axis=1)
    return X4d, y

In [9]:
# Build the leak-free FBCSP pipeline and stress-test across subjects
def evaluate_fbcsp(subject, n_csp=4, k_features=None):
    raw_s, evs, ev_id = load_subject_raw(subject)
    X4d, y = build_band_data(raw_s, evs, ev_id)

    steps = [('FBCSP', FBCSP(n_csp=n_csp))]
    if k_features is not None:
        steps.append(('select', SelectKBest(mutual_info_classif, k=k_features)))
    steps.append(('LDA', LinearDiscriminantAnalysis()))
    pipe = Pipeline(steps)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X4d, y, cv=cv, scoring='accuracy')
    return scores.mean(), scores.std()

# Stress test: compare FBCSP against your baseline across several subjects
test_subjects = [1, 2, 3, 4, 5, 6, 7, 8]
print(f"{'Subj':>5} | {'FBCSP acc':>10} | {'std':>6}")
print("-" * 28)
for s in test_subjects:
    try:
        m, sd = evaluate_fbcsp(s, n_csp=4, k_features=None)   # select best 8 of 28 features
        print(f"{s:>5} | {m:>10.3f} | {sd:>6.3f}")
    except Exception as e:
        print(f"{s:>5} | FAILED: {type(e).__name__}")

 Subj |  FBCSP acc |    std
----------------------------
Computing rank from data with rank=None
    Using tolerance 0.00014 (2.2e-16 eps * 64 dim * 9.7e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 64 dim * 8.5e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 9.8e-05 (2.2e-16 eps * 64 dim * 6.9e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data r

In [10]:
# Diagnostic: FBCSP without feature selection AND without the noisy 4-8 Hz band
filter_bank_clean = [(8,12),(12,16),(16,20),(20,24),(24,28),(28,32)]  # dropped 4-8

def evaluate_fbcsp_clean(subject, n_csp=4):
    global filter_bank
    saved = filter_bank
    filter_bank = filter_bank_clean          # temporarily use clean bank
    try:
        raw_s, evs, ev_id = load_subject_raw(subject)
        X4d, y = build_band_data(raw_s, evs, ev_id)
        pipe = Pipeline([('FBCSP', FBCSP(n_csp=n_csp)),
                         ('LDA', LinearDiscriminantAnalysis())])   # NO feature selection
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(pipe, X4d, y, cv=cv, scoring='accuracy')
    finally:
        filter_bank = saved                   # restore
    return scores.mean(), scores.std()

for s in [2, 1, 3]:   # strong, medium, weak
    m, sd = evaluate_fbcsp_clean(s)
    print(f"S{s}: {m:.3f} (±{sd:.3f})")

Computing rank from data with rank=None
    Using tolerance 7.8e-05 (2.2e-16 eps * 64 dim * 5.5e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 6.8e-05 (2.2e-16 eps * 64 dim * 4.8e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 4.6e-05 (2.2e-16 eps * 64 dim * 3.2e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMP

In [11]:
# DECISIVE TEST: run the FBCSP machinery with ONE band = baseline's 8-30 Hz
# This should EXACTLY reproduce the plain CSP+LDA baseline (~0.867 for subj 2)
filter_bank_single = [(8, 30)]   # single wide band = the baseline

def evaluate_fbcsp_custombank(subject, bank, n_csp=6):
    global filter_bank
    saved = filter_bank
    filter_bank = bank
    try:
        raw_s, evs, ev_id = load_subject_raw(subject)
        X4d, y = build_band_data(raw_s, evs, ev_id)
        pipe = Pipeline([('FBCSP', FBCSP(n_csp=n_csp)),
                         ('LDA', LinearDiscriminantAnalysis())])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(pipe, X4d, y, cv=cv, scoring='accuracy')
    finally:
        filter_bank = saved
    return scores.mean(), scores.std()

# Subject 2, single 8-30 band, 6 components (matching your baseline exactly)
m, sd = evaluate_fbcsp_custombank(2, filter_bank_single, n_csp=6)
print(f"Single-band (8-30) through FBCSP machinery: {m:.3f} (±{sd:.3f})")
print("Baseline CSP+LDA was: 0.867")
print("If these match -> machinery is correct, multi-band just doesn't help")
print("If these differ -> there's a bug in the FBCSP transformer")

Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 64 dim * 7.2e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 9.9e-05 (2.2e-16 eps * 64 dim * 7e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 64 dim * 7.1e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRIC

# Log: FBCSP investigation — validated, but doesn't help on this data

## What I did
- Implemented Filter Bank CSP (FBCSP): 7 bands (4-32 Hz), per-band CSP, feature
  stacking, mutual-information feature selection, LDA.
- Stress-tested across subjects 1-8 against the single-band CSP+LDA baseline.

## What I found
- FBCSP consistently matched or UNDERPERFORMED the baseline. Subject 2 dropped
  from 0.867 (baseline) to ~0.69 with FBCSP.
- Ruled out feature selection and the noisy 4-8 Hz band as causes (removing them
  didn't recover performance).
- Decisive test: ran the FBCSP machinery with a single 8-30 Hz band — it exactly
  reproduced the 0.867 baseline. This PROVES the code is mechanically correct.

## Why it underperforms (the real reason)
- Not a bug. Multi-band splitting genuinely hurts on this small-data regime.
- Each narrow 4 Hz band gives CSP only a thin slice of signal, so per-band
  covariance estimates (64x64 from ~36 training trials) are noisier.
- Stacking 28 noisy features and fitting LDA with only 45 trials overfits.
- Single wide-band CSP captures the full motor signal in one clean, well-estimated
  representation — better with limited data.

## Conclusion
- Kept single-band CSP+LDA as the validated pipeline.
- Lesson: method complexity must be justified by data availability. FBCSP won BCI
  competitions on datasets with more trials/subject than PhysioNet provides here.
- This is the second honest negative result (after window-tuning overfitting):
  the simpler method is the right choice for this regime, for understood reasons.

In [12]:
import sys
sys.path.append(str(__import__('pathlib').Path.home() / "projects" / "bci"))
from src import config
print("Band:", config.BAND_LOW, "-", config.BAND_HIGH, "Hz")
print("Window:", config.EPOCH_TMIN, "-", config.EPOCH_TMAX, "s")
print("Components:", config.N_CSP_COMPONENTS)
print("Config imported OK")

Band: 8.0 - 30.0 Hz
Window: 0.5 - 2.5 s
Components: 4
Config imported OK


In [13]:
from src import data_loading

# Load subject 2 (already downloaded, should be instant)
raw, events, event_id = data_loading.load_subject_raw(2)
print("Raw:", raw)
print("Duration:", round(raw.times[-1], 1), "s")
print("Channels:", len(raw.ch_names))
print("Event IDs:", event_id)
print("data_loading OK")


Raw: <RawEDF | S002R04.edf, 64 x 59040 (369.0 s), ~28.9 MiB, data loaded>
Duration: 369.0 s
Channels: 64
Event IDs: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
data_loading OK


In [15]:
from src import data_loading, preprocessing

raw, events, event_id = data_loading.load_subject_raw(2)
epochs = preprocessing.make_epochs(raw, events, event_id)
X, y = preprocessing.get_Xy(epochs)

print("Epochs:", epochs)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Label counts:", {int(k): int((y==k).sum()) for k in set(y)})
print("preprocessing OK")

Epochs: <Epochs | 45 events (all good), 0.5 – 2.5 s (baseline off), ~7.1 MiB, data loaded,
 'T1': 23
 'T2': 22>
X shape: (45, 64, 321)
y shape: (45,)
Label counts: {2: 23, 3: 22}
preprocessing OK


In [16]:
from src import pipeline

# Full end-to-end evaluation of subject 2 through the refactored modules
result = pipeline.evaluate_subject(2)
print(f"Subject 2 accuracy: {result['mean']:.3f} (±{result['std']:.3f})")
print(f"Chance level: {result['chance']:.3f}")
print(f"Per-fold: {result['scores'].round(3)}")

Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 64 dim * 7.2e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 9.9e-05 (2.2e-16 eps * 64 dim * 7e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 64 dim * 7.1e+09  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRIC